In [1]:
from pickleshare import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE #due to class imbalance in the target column
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle

In [4]:
df = pd.read_excel("/content/Lloyds_customer_churn_ML.xlsx")
print(df.head())
df.info()
df.shape

   CustomerID  Age Gender MaritalStatus IncomeLevel  Total_amount_spent  \
0           1   62      M        Single         Low              416.50   
1           2   65      M       Married         Low             1547.42   
2           3   18      M        Single         Low             1702.98   
3           4   21      M       Widowed         Low              917.29   
4           5   21      M      Divorced      Medium             2001.49   

   Days_from_last_transaction_date_2022  Days_from_last_interaction_date_2022  \
0                                   279                                   275   
1                                    42                                   289   
2                                    84                                   129   
3                                     4                                    43   
4                                    10                                 44926   

   complaint  feedback  inquiries  ResolvedInquiry  Unresolved

(1000, 16)

In [5]:
df = df.drop(columns = ["CustomerID"])

In [8]:
#encode features that are still categorised in text format (object)
object_columns =df.select_dtypes(include="object").columns
print(object_columns)

Index(['Gender', 'MaritalStatus', 'IncomeLevel'], dtype='object')


In [9]:
#initialize a dictionary to save encoders
encoders ={}

#apply label encodinng and store the encoders
for column in object_columns:
  label_encoders = LabelEncoder()
  df[column] = label_encoders.fit_transform(df[column])
  encoders[column] = label_encoders

#save the encoders to a picle file
with open("encoders.pkl", "wb") as f:
  pickle.dump(encoders, f)

In [10]:
encoders

{'Gender': LabelEncoder(),
 'MaritalStatus': LabelEncoder(),
 'IncomeLevel': LabelEncoder()}

In [11]:
df.head()

,Age,Gender,MaritalStatus,IncomeLevel,Total_amount_spent,Days_from_last_transaction_date_2022,Days_from_last_interaction_date_2022,complaint,feedback,inquiries,ResolvedInquiry,UnresolvedInquiry,Days_from_last_login_date_2023,AverageLoginFrequency,ChurnStatus
0,62,1,2,1,416.50,279,275,0,0,1,1,0,71,34,0
1,65,1,1,1,1547.42,42,289,0,0,1,1,0,26,5,1
2,18,1,2,1,1702.98,84,129,0,0,1,1,0,46,3,0
3,21,1,3,1,917.29,4,43,0,0,2,1,1,128,2,0
4,21,1,0,2,2001.49,10,44926,0,0,0,0,0,65,41,0


In [12]:
#EDA - check the class distribution of the target column
print(df["ChurnStatus"].value_counts())

ChurnStatus
0    796
1    204
Name: count, dtype: int64


In [13]:
#split feature from target
x=df.drop(columns=["ChurnStatus"])
y=df["ChurnStatus"]

In [14]:
from os import X_OK
#train test split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify = y, random_state=42)

In [19]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.pipeline import Pipeline

In [24]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42)
}


In [25]:
from sklearn import pipeline
cv_scores = {}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for model_name, model in models.items():
    print(f"Training {model_name}")

    pipeline = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model) ])

    scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring="roc_auc")

    cv_scores[model_name]= scores
    print(f"{model_name} mean ROC-AUC: {np.mean(scores):.3f}")
    print("-"*50)

Training Decision Tree
Decision Tree mean ROC-AUC: 0.518
--------------------------------------------------
Training Random Forest
Random Forest mean ROC-AUC: 0.516
--------------------------------------------------
Training XGBoost
XGBoost mean ROC-AUC: 0.579
--------------------------------------------------


In [26]:
#xgb performed best
xgb_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.05, 0.1]
}

grid = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train, y_train)


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('smote', SMOTE(random_state=42)),
                                       ('model',
                                        XGBClassifier(base_score=None,
                                                      booster=None,
                                                      callbacks=None,
                                                      colsample_bylevel=None,
                                                      colsample_bynode=None,
                                                      colsample_bytree=None,
                                                      device=None,
                                                      early_stopping_rounds=None,
                                                      enable_categorical=False,
                                                      eval_metric=None,
                                                      fea...
                                                      max_cat_threshold=None,
                                                      max_cat_to_onehot=None,
                                                      max_delta_step=None,
                                                      max_depth=None,
                                                      max_leaves=None,
                                                      min_child_weight=None,
                                                      missing=nan,
                                                      monotone_constraints=None,
                                                      multi_strategy=None,
                                                      n_estimators=None,
                                                      n_jobs=None,
                                                      num_parallel_tree=None, ...))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.05, 0.1],
                         'model__max_depth': [3, 5, 7],
                         'model__n_estimators': [100, 200]},
             scoring='roc_auc')

In [28]:
print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)


Best params: {'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 200}
Best CV ROC-AUC: 0.5722034568122167


In [30]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
#print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.81      0.84      0.82       159
           1       0.26      0.22      0.24        41

    accuracy                           0.71       200
   macro avg       0.54      0.53      0.53       200
weighted avg       0.70      0.71      0.70       200

Confusion Matrix:
 [[134  25]
 [ 32   9]]


In [32]:
print(y_test.value_counts())


ChurnStatus
0    159
1     41
Name: count, dtype: int64


Hyperparameter tuning didn't work either.


I should have done this at the beginning, but, when feature correlation with target is low, it means the data has weak predictive power and I checked this using the code below. Feature engineering is more powerful than model tuning.

In [34]:
df.corr()['ChurnStatus'].sort_values()


,ChurnStatus
AverageLoginFrequency,-0.081615
Days_from_last_interaction_date_2022,-0.035075
inquiries,-0.022154
MaritalStatus,0.000270
ResolvedInquiry,0.001209
Total_amount_spent,0.001324
Days_from_last_transaction_date_2022,0.003362
UnresolvedInquiry,0.005132
IncomeLevel,0.008134
Days_from_last_login_date_2023,0.009055


Next step/ Improvement
1. Feature engineering: Adjusting the features to bring the best out of them